In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [11]:

# Your token (better: store in .env and load it securely)
token = "YOUR_GITHUB_TOKEN_HERE"

url = "https://raw.githubusercontent.com/SubharupBiswas/sih2025/main/crop_yield_dataset.csv"
headers = {"Authorization": f"token {token}"}

response = requests.get(url, headers=headers)

df = pd.read_csv(StringIO(response.text))
print(df)   # (rows, columns)



             Date  Crop_Type Soil_Type  Soil_pH  Temperature   Humidity  \
0      2014-01-01      Wheat     Peaty     5.50     9.440599  80.000000   
1      2014-01-01       Corn     Loamy     6.50    20.052576  79.947424   
2      2014-01-01       Rice     Peaty     5.50    12.143099  80.000000   
3      2014-01-01     Barley     Sandy     6.75    19.751848  80.000000   
4      2014-01-01    Soybean     Peaty     5.50    16.110395  80.000000   
...           ...        ...       ...      ...          ...        ...   
36515  2023-12-31     Cotton      Clay     6.25    19.538555  80.000000   
36516  2023-12-31  Sugarcane     Peaty     5.50    21.068336  78.931664   
36517  2023-12-31     Tomato     Sandy     6.75     6.030148  80.000000   
36518  2023-12-31     Potato     Peaty     5.50    11.079561  80.000000   
36519  2023-12-31  Sunflower      Clay     6.25    11.455692  80.000000   

       Wind_Speed     N     P     K  Crop_Yield  Soil_Quality  
0       10.956707  60.5  45.0  31.5

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isnull().sum())

before_dupes = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
after_dupes = df.shape[0]
print(f"\nDropped duplicates: {before_dupes - after_dupes}")

In [19]:
date_cols = [c for c in df.columns if "date" in c.lower()]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")
    df[f"{c}_Year"] = df[c].dt.year
    df[f"{c}_Month"] = df[c].dt.month
    df[f"{c}_DayOfYear"] = df[c].dt.dayofyear
    df[f"{c}_Week"] = df[c].dt.isocalendar().week.astype("Int64")
    df[f"{c}_Quarter"] = df[c].dt.quarter

In [21]:
possible_targets = ["Crop_Yield", "Yield", "yield", "target", "Target"]
target_col = next((c for c in possible_targets if c in df.columns), None)
if target_col is None:
    raise ValueError("Target column not found. Expected one of: " + ", ".join(possible_targets))

In [22]:
# 8) Split features (X) and target (y); drop raw date columns from features to avoid leakage
X = df.drop(columns=[target_col] + date_cols)
y = df[target_col]

In [23]:
# 9) Identify numeric and categorical feature columns
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)


Numeric features: ['Soil_pH', 'Temperature', 'Humidity', 'Wind_Speed', 'N', 'P', 'K', 'Soil_Quality', 'Date_Year_Year', 'Date_Year_Month', 'Date_Year_DayOfYear', 'Date_Year_Week', 'Date_Year_Quarter', 'Date_Month_Year', 'Date_Month_Month', 'Date_Month_DayOfYear', 'Date_Month_Week', 'Date_Month_Quarter', 'Date_DayOfYear_Year', 'Date_DayOfYear_Month', 'Date_DayOfYear_DayOfYear', 'Date_DayOfYear_Week', 'Date_DayOfYear_Quarter', 'Date_Week_Year', 'Date_Week_Month', 'Date_Week_DayOfYear', 'Date_Week_Week', 'Date_Week_Quarter', 'Date_Quarter_Year', 'Date_Quarter_Month', 'Date_Quarter_DayOfYear', 'Date_Quarter_Week', 'Date_Quarter_Quarter']
Categorical features: ['Crop_Type', 'Soil_Type']


In [28]:
# 10) Build preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))  # Changed 'sparse' to 'sparse_output'
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [29]:
# 11) Fit and transform the dataset
X_processed = preprocess.fit_transform(X)

In [30]:
# 12) Recover feature names after transformation
num_feature_names = numeric_features
cat_feature_names = []
if len(categorical_features) > 0:
    cat_feature_names = list(
        preprocess.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
    )
final_feature_names = num_feature_names + cat_feature_names

In [31]:
# 13) Wrap into a processed DataFrame
X_processed_df = pd.DataFrame(X_processed, columns=final_feature_names, index=X.index)

In [33]:
# 14) Train-test split (stratify not used for regression)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed_df, y, test_size=0.2, random_state=42, shuffle=True
)

print("\nProcessed shapes:")
print("X_train:", X_train.shape, "X_test:", X_test.shape, "y_train:", y_train.shape, "y_test:", y_test.shape)


Processed shapes:
X_train: (29216, 48) X_test: (7304, 48) y_train: (29216,) y_test: (7304,)


In [34]:
# 15) Save processed datasets to disk
train_out = Path("/mnt/data/processed_train.csv")
test_out = Path("/mnt/data/processed_test.csv")
y_train_out = Path("/mnt/data/processed_y_train.csv")
y_test_out = Path("/mnt/data/processed_y_test.csv")

X_train.to_csv(train_out, index=False)
X_test.to_csv(test_out, index=False)
pd.DataFrame({"Crop_Yield": y_train}).to_csv(y_train_out, index=False)
pd.DataFrame({"Crop_Yield": y_test}).to_csv(y_test_out, index=False)

print("\nSaved:")
print(" -", train_out)
print(" -", test_out)
print(" -", y_train_out)
print(" -", y_test_out)

OSError: Cannot save file into a non-existent directory: '\mnt\data'